<h1 align = "center", style = "color:green;">Random Forest Programming<h1> 

# With Python

In [1]:
# Import important Libraries
import numpy as np

In [2]:
# Defining Gini Index
def gini(y):
    classes = np.unique(y)
    impurity = 1

    for c in classes:
        p = np.sum(y == c) / len(y)
        impurity -= p**2
        
    return impurity

In [3]:
# Defining Random Forest Best Split
def best_split_rf(X, y, n_features_to_try):
    best_feature = None
    best_threshold = None
    best_gini = 1

    n_samples, n_features = X.shape

    # RANDOM subset of features
    features = np.random.choice(
        n_features, 
        n_features_to_try, 
        replace=False
    )

    for feature in features:
        thresholds = np.unique(X[:, feature])

        for t in thresholds:
            left = y[X[:, feature] <= t]
            right = y[X[:, feature] > t]

            if len(left)==0 or len(right)==0:
                continue

            g = (len(left)/len(y))*gini(left) \
                + (len(right)/len(y))*gini(right)

            if g < best_gini:
                best_gini = g
                best_feature = feature
                best_threshold = t

    return best_feature, best_threshold

In [4]:
# Defining Node
class Node:
    def __init__(
        self, 
        feature=None, 
        threshold=None, 
        left=None, 
        right=None, 
        value=None
    ):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

In [ ]:
# Defining Random Forest Build Tree
def build_tree_rf(
    X, 
    y, 
    depth, 
    max_depth, 
    n_features_to_try
):
    if len(np.unique(y)) == 1 or depth == max_depth:
        return Node(value=np.bincount(y).argmax())

    feature, threshold = best_split_rf(
        X, 
        y, 
        n_features_to_try
    )

    if feature is None:
        return Node(value=np.bincount(y).argmax())

    left_idx = X[:, feature] <= threshold
    right_idx = X[:, feature] > threshold

    left = build_tree_rf(
        X[left_idx], 
        y[left_idx], 
        depth+1, 
        max_depth, 
        n_features_to_try
    )
    right = build_tree_rf(
        X[right_idx], 
        y[right_idx], 
        depth+1, 
        max_depth, 
        n_features_to_try
    )

    return Node(
        feature, 
        threshold, 
        left, 
        right
    )

In [ ]:
# Defining Bootstrap Sampling
def bootstrap_sample(X, y):
    n = len(X)
    idx = np.random.choice(n, n, replace=True)
    
    return X[idx], y[idx]

In [ ]:
# Defining Random Forest
class RandomForest:
    def __init__(self, n_trees=10, max_depth=4):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        n_features_to_try = int(np.sqrt(X.shape[1]))

        for _ in range(self.n_trees):
            X_s, y_s = bootstrap_sample(X, y)

            tree = build_tree_rf(
                X_s, 
                y_s, 
                0, 
                self.max_depth, 
                n_features_to_try
            )
            self.trees.append(tree)

    def predict(self, X):
        all_preds = []

        for tree in self.trees:
            preds = np.array(
                [predict_one(tree, x) for x in X]
            )
            all_preds.append(preds)

        # majority vote
        all_preds = np.array(all_preds)
        return np.round(np.mean(all_preds, axis=0)) \
            .astype(int)

In [ ]:
# Prediction
def predict_one(node, x):
    if node.value is not None:
        return node.value

    if x[node.feature] <= node.threshold:
        return predict_one(node.left, x)
        
    else:
        return predict_one(node.right, x)

def predict(tree, X):
    return np.array(
        [predict_one(tree, x) for x in X]
    )

In [ ]:
# Model Training and Prediction
X = np.array([[2],[3],[4],[5],[6],[7],[8],[9]])
y = np.array([0,0,0,0,1,1,1,1])

rf = RandomForest(n_trees=20, max_depth=3)
rf.fit(X,y)

preds = rf.predict(X)
print(preds)

[0 0 0 0 1 1 1 1]
